# NB03 — Quality Control & Tissue Masking

Per-slide tissue percentage, blur (Laplacian variance), pen-marking detection, and brightness/saturation summaries computed on a 1024-px thumbnail using HSV-based segmentation. Slides failing any threshold are flagged for exclusion. Outputs include per-slide QC parquet, an exclusions CSV, thumbnail+overlay JPEGs, four summary diagnostic figures, and a compute-passport entry.

In [ ]:
import os, sys, time, json, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import openslide

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
SUBDIRS = {
    'compute':   WORKSPACE / 'compute',
    'logs':      WORKSPACE / 'logs',
    'figures':   WORKSPACE / 'figures',
    'qc':        WORKSPACE / 'qc',
    'manifests': WORKSPACE / 'manifests',
}
for p in SUBDIRS.values():
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_PARQUET   = SUBDIRS['manifests'] / 'manifest_tcga.parquet'
QC_METRICS_PARQUET = SUBDIRS['qc'] / 'qc_metrics_tcga.parquet'
QC_METRICS_CSV     = SUBDIRS['qc'] / 'qc_metrics_tcga.csv'
QC_EXCLUSIONS_CSV  = SUBDIRS['qc'] / 'exclusions_tcga.csv'
QC_THUMBS_DIR      = SUBDIRS['qc'] / 'thumbs'
FIG_DIR            = SUBDIRS['figures']
QC_THUMBS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PARQUET.exists(), f'manifest not found at {MANIFEST_PARQUET}; run NB02 first'

QC_MAX_SLIDES   = None
THUMB_MAX_SIDE  = 1024
MIN_TISSUE_PCT  = 0.10
MAX_WHITE_PCT   = 0.75
MIN_BLUR_VAR    = 15.0
MAX_PEN_PCT     = 0.02
HSV_S_TISSUE_MIN = 20
HSV_V_WHITE_MIN  = 230
HSV_S_WHITE_MAX  = 30
HSV_H_BLUE_MIN   = 170
HSV_H_BLUE_MAX   = 255
HSV_S_PEN_MIN    = 60
MAX_WORKERS = min(8, (os.cpu_count() or 8))

def now_iso():
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def load_thumbnail(slide_path: Path, max_side: int = 1024) -> Image.Image:
    slide = openslide.OpenSlide(str(slide_path))
    w, h = slide.dimensions
    scale = max(w, h) / max_side if max(w, h) > max_side else 1.0
    tw, th = int(w / scale), int(h / scale)
    thumb = slide.get_thumbnail((tw, th)).convert('RGB')
    slide.close()
    return thumb

def to_hsv_np(img_rgb: Image.Image):
    hsv = img_rgb.convert('HSV')
    a = np.array(hsv, dtype=np.uint8)
    return a[..., 0], a[..., 1], a[..., 2]

def laplacian_var(gray_u8: np.ndarray, tissue_mask: np.ndarray = None) -> float:
    g = gray_u8.astype(np.float32)
    p = np.pad(g, 1, mode='reflect')
    c = -4 * p[1:-1, 1:-1]
    n = 1 * (p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:])
    lap = c + n
    if tissue_mask is not None:
        mask = tissue_mask.astype(bool)
        if mask.sum() == 0:
            return 0.0
        vals = lap[mask]
    else:
        vals = lap.ravel()
    return float(np.var(vals))

def qc_on_thumbnail(img: Image.Image):
    H, S, V = to_hsv_np(img)
    gray = np.array(img.convert('L'), dtype=np.uint8)
    tissue_mask = (S >= HSV_S_TISSUE_MIN) & (V < HSV_V_WHITE_MIN)
    white_mask  = (V >= HSV_V_WHITE_MIN) & (S <= HSV_S_WHITE_MAX)
    pen_mask    = (H >= HSV_H_BLUE_MIN) & (H <= HSV_H_BLUE_MAX) & (S >= HSV_S_PEN_MIN)
    total = img.size[0] * img.size[1]
    metrics = {
        'tissue_pct': float(tissue_mask.sum() / total),
        'white_pct':  float(white_mask.sum() / total),
        'pen_pct':    float(pen_mask.sum() / total),
        'blur_var':   laplacian_var(gray, tissue_mask),
    }
    if tissue_mask.sum() > 0:
        metrics['brightness_mean'] = float(V[tissue_mask].mean())
        metrics['saturation_mean'] = float(S[tissue_mask].mean())
    else:
        metrics['brightness_mean'] = float(V.mean())
        metrics['saturation_mean'] = float(S.mean())
    return metrics, tissue_mask

def qc_reason_flags(m, thresholds):
    reasons = []
    if m['tissue_pct'] < thresholds['min_tissue_pct']:
        reasons.append(f"low_tissue<{thresholds['min_tissue_pct']:.2f}")
    if m['white_pct'] > thresholds['max_white_pct']:
        reasons.append(f"white>{thresholds['max_white_pct']:.2f}")
    if m['blur_var'] < thresholds['min_blur_var']:
        reasons.append(f"blur<{thresholds['min_blur_var']:.1f}")
    if m['pen_pct'] > thresholds['max_pen_pct']:
        reasons.append(f"pen>{thresholds['max_pen_pct']:.2f}")
    return reasons

def save_thumb_and_mask(slide_id: str, img: Image.Image, tissue_mask: np.ndarray):
    thumb_path = QC_THUMBS_DIR / f'{slide_id}_thumb.jpg'
    img.save(str(thumb_path), 'JPEG', quality=90)
    overlay = np.array(img).copy()
    red = np.zeros_like(overlay); red[..., 0] = 255
    alpha = 0.35
    mask3 = np.stack([tissue_mask]*3, axis=-1)
    overlay = (overlay * (~mask3) + (alpha * overlay + (1 - alpha) * red) * mask3).astype(np.uint8)
    overlay_path = QC_THUMBS_DIR / f'{slide_id}_overlay.jpg'
    Image.fromarray(overlay).save(str(overlay_path), 'JPEG', quality=90)
    return str(thumb_path), str(overlay_path)

print(f'[{now_iso()}] loading manifest: {MANIFEST_PARQUET}')
df_manifest = pd.read_parquet(MANIFEST_PARQUET).copy()
if QC_MAX_SLIDES is not None:
    df_manifest = df_manifest.head(QC_MAX_SLIDES).copy()
print(f'[INFO] slides to QC: {len(df_manifest)}')

thresholds = {
    'min_tissue_pct': MIN_TISSUE_PCT,
    'max_white_pct': MAX_WHITE_PCT,
    'min_blur_var': MIN_BLUR_VAR,
    'max_pen_pct': MAX_PEN_PCT,
}

results = []; failures = []
t_start = time.time()

def worker(row):
    slide_path = Path(row['path'])
    slide_id   = str(row['slide_id'])
    cancer     = str(row.get('cancer_code', 'UNKNOWN'))
    try:
        img = load_thumbnail(slide_path, THUMB_MAX_SIDE)
        metrics, tissue_mask = qc_on_thumbnail(img)
        reasons = qc_reason_flags(metrics, thresholds)
        thumb_p, overlay_p = save_thumb_and_mask(slide_id, img, tissue_mask)
        rec = {
            'slide_id': slide_id, 'cancer_code': cancer, 'path': str(slide_path),
            'tissue_pct': metrics['tissue_pct'], 'white_pct': metrics['white_pct'],
            'pen_pct': metrics['pen_pct'], 'blur_var': metrics['blur_var'],
            'brightness_mean': metrics['brightness_mean'],
            'saturation_mean': metrics['saturation_mean'],
            'excluded': int(len(reasons) > 0),
            'reasons': ';'.join(reasons) if reasons else '',
            'thumb': thumb_p, 'overlay': overlay_p,
        }
        return True, rec
    except Exception as e:
        return False, {'slide_id': slide_id, 'path': str(slide_path), 'error': f'{e.__class__.__name__}: {e}'}

done = 0; last_print = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(worker, row) for _, row in df_manifest.iterrows()]
    for fut in as_completed(futs):
        ok, payload = fut.result()
        if ok: results.append(payload)
        else:  failures.append(payload)
        done += 1
        now = time.time()
        if now - last_print > 2 or done == len(df_manifest):
            rate = done / (now - t_start + 1e-9)
            print(f'  QC {done}/{len(df_manifest)} ({rate:.1f} slides/s)')
            last_print = now

elapsed = time.time() - t_start
print(f'[OK] QC completed in {elapsed/60:.1f} min')

df_qc = pd.DataFrame.from_records(results).sort_values('slide_id').reset_index(drop=True)
df_qc.to_parquet(QC_METRICS_PARQUET, index=False)
df_qc.to_csv(QC_METRICS_CSV, index=False, encoding='utf-8-sig')

if failures:
    pd.DataFrame(failures).to_csv(SUBDIRS['qc'] / 'qc_failures.csv', index=False, encoding='utf-8-sig')

df_excl = df_qc[df_qc['excluded'] == 1].copy()
df_excl.to_csv(QC_EXCLUSIONS_CSV, index=False, encoding='utf-8-sig')

n_total = len(df_qc); n_excl = len(df_excl)
print(f'\n  total slides QC\'d: {n_total:,}')
print(f'  excluded:          {n_excl:,} ({100.0*n_excl/max(1, n_total):.1f}%)')
if n_excl > 0:
    reason_counter = {}
    for r in df_excl['reasons']:
        for tok in str(r).split(';'):
            tok = tok.strip()
            if tok:
                reason_counter[tok] = reason_counter.get(tok, 0) + 1
    print('  exclusion reasons:')
    for k, v in sorted(reason_counter.items(), key=lambda x: -x[1]):
        print(f'    {k}: {v}')

# four summary figures matching the manifest figure pattern
plt.figure(figsize=(8,5))
plt.hist(df_qc['tissue_pct'].dropna().values, bins=40)
plt.axvline(MIN_TISSUE_PCT, color='red', ls='--', label=f'threshold {MIN_TISSUE_PCT:.2f}')
plt.xlabel('Tissue fraction'); plt.ylabel('Slides')
plt.title('Tissue percentage per slide')
plt.legend(); plt.tight_layout()
p1 = FIG_DIR / 'qc_tissue_pct.png'
plt.savefig(p1, dpi=200); plt.close()

plt.figure(figsize=(8,5))
plt.hist(df_qc['blur_var'].dropna().values, bins=40)
plt.axvline(MIN_BLUR_VAR, color='red', ls='--', label=f'threshold {MIN_BLUR_VAR}')
plt.xlabel('Laplacian variance'); plt.ylabel('Slides')
plt.title('Blur (Laplacian variance) on tissue regions')
plt.legend(); plt.tight_layout()
p2 = FIG_DIR / 'qc_blur_distribution.png'
plt.savefig(p2, dpi=200); plt.close()

plt.figure(figsize=(8,5))
plt.hist(df_qc['white_pct'].dropna().values, bins=40)
plt.axvline(MAX_WHITE_PCT, color='red', ls='--', label=f'threshold {MAX_WHITE_PCT:.2f}')
plt.xlabel('White fraction'); plt.ylabel('Slides')
plt.title('White-area fraction per slide')
plt.legend(); plt.tight_layout()
p3 = FIG_DIR / 'qc_white_distribution.png'
plt.savefig(p3, dpi=200); plt.close()

if 'cancer_code' in df_qc.columns:
    excl_by_cancer = df_qc.groupby('cancer_code')['excluded'].agg(['sum', 'count']).rename(
        columns={'sum': 'excluded', 'count': 'total'})
    excl_by_cancer['rate'] = excl_by_cancer['excluded'] / excl_by_cancer['total'].clip(lower=1)
    excl_by_cancer = excl_by_cancer.sort_values('rate', ascending=False).head(30)
    plt.figure(figsize=(10,6))
    plt.bar(excl_by_cancer.index.astype(str), 100 * excl_by_cancer['rate'].values)
    plt.xticks(rotation=80, ha='right')
    plt.ylabel('Exclusion rate (%)')
    plt.title('Exclusion rate by cancer code (top 30)')
    plt.tight_layout()
    p4 = FIG_DIR / 'qc_exclusion_by_cancer.png'
    plt.savefig(p4, dpi=200); plt.close()
else:
    p4 = None

compute_path = SUBDIRS['compute'] / 'compute_passport.json'
try:
    with compute_path.open('r', encoding='utf-8') as f:
        cp = json.load(f)
except Exception:
    cp = {'stages': []}
stage_entry = {
    'stage': 'qc_tcga', 'timestamp': now_iso(),
    'inputs': {'manifest_parquet': str(MANIFEST_PARQUET)},
    'outputs': {
        'qc_metrics_parquet': str(QC_METRICS_PARQUET),
        'qc_metrics_csv':     str(QC_METRICS_CSV),
        'exclusions_csv':     str(QC_EXCLUSIONS_CSV),
        'thumbs_dir':         str(QC_THUMBS_DIR),
        'figures':            [str(p1), str(p2), str(p3), str(p4) if p4 else None],
    },
    'thresholds': thresholds,
    'stats': {
        'n_records': int(len(df_qc)),
        'n_excluded': int(len(df_excl)),
        'exclusion_rate': float(len(df_excl) / max(1, len(df_qc))),
        'elapsed_minutes': float(elapsed / 60.0),
    },
}
cp.setdefault('stages', []).append(stage_entry)
tmp = compute_path.parent / (compute_path.name + '.tmp')
with tmp.open('w', encoding='utf-8') as f:
    json.dump(cp, f, ensure_ascii=False, indent=2)
tmp.replace(compute_path)
print(f'\n[OK] compute passport updated: {compute_path}')
print('NB03 complete. Next: NB04 (two-scale tiling).')